# TurboVLA / Evo-1 — Smoke チェックリスト

教材: [`../04_upstream_smoke.md`](../04_upstream_smoke.md)  
調査メモ: [`../../parc/docs/00_research/turbovla_evo1.md`](../../parc/docs/00_research/turbovla_evo1.md)

**目的:** フル学習ではなく、server 起動や少数 trial eval（または GPU 無しウォークスルー）を記録する。

このノートはパスを埋めるテンプレです。巨大依存の自動 install は行いません。


## 0. メタ情報（必ず埋める）

| 項目 | 記入欄 |
|------|--------|
| 日付 | |
| 実行ホスト | thor / winpc / nuc / ローカル 他: |
| GPU 有無 | あり / なし |
| 実施したトラック | Evo-1 / TurboVLA / 両方 / ウォークスルーのみ |
| 結果サマリ（1行） | |


In [ ]:
from pathlib import Path
from datetime import datetime

# === 学習者が編集 ===
HOST = "thor"  # 例: thor / winpc / local
HAS_GPU = True
NOTES = ""

print("recorded_at:", datetime.now().isoformat(timespec="seconds"))
print("host:", HOST, "has_gpu:", HAS_GPU)
print("notes:", NOTES or "(none)")


## 1. Evo-1 — パス確認

thor 想定の既定値。環境が違う場合は上書きする。


In [ ]:
from pathlib import Path

EVO1_CLONE = Path("/mnt/sda/parc_libero_plus/third_party/Evo-1")
EVO1_CKPT = Path("/mnt/sda/parc_libero_plus/checkpoints/Evo1_LIBERO")

for label, p in [("clone", EVO1_CLONE), ("ckpt", EVO1_CKPT)]:
    print(f"{label}: {p}  exists={p.exists()}")
    if p.exists():
        # 上位だけ列挙（巨大ツリーを全 dump しない）
        kids = sorted(p.iterdir())[:12]
        print("  sample:", [k.name for k in kids])


### Evo-1 server smoke

ターミナルで（ノートからは起動しない想定）:

```bash
cd "$EVO1_CLONE/Evo_1"
# Evo1_server.py の checkpoint パスを EVO1_CKPT に合わせる
python scripts/Evo1_server.py
```

| チェック | 結果 (Y/N/スキップ) | メモ |
|----------|---------------------|------|
| clone / ckpt が存在する | | |
| server が bind / ready 相当を出した | | |
| （任意）libero-plus client が接続した | | |
| 失敗理由（GPU 競合など） | | |


## 2. TurboVLA — パスと引数メモ


In [ ]:
from pathlib import Path

# 学習者が編集: クローン先・HF ダウンロード先
TURBOVLA_ROOT = Path("CHANGE_ME/TurboVLA")
TURBOVLA_HF = Path("CHANGE_ME/pretrained/TurboVLA")

print("root exists:", TURBOVLA_ROOT.exists(), TURBOVLA_ROOT)
print("hf   exists:", TURBOVLA_HF.exists(), TURBOVLA_HF)

# プロトコル契約（評価時は一致させる）
CHUNK_SIZE = 12
NUM_OPEN_LOOP_STEPS = 12
STATS_KEY = "libero_all4_no_noops"
assert CHUNK_SIZE == NUM_OPEN_LOOP_STEPS, "chunk と open-loop を一致させる"
print("ok:", {"chunk_size": CHUNK_SIZE, "num_open_loop_steps": NUM_OPEN_LOOP_STEPS, "stats_key": STATS_KEY})


### TurboVLA eval smoke（GPU あり）

公式に近いコマンドの骨格（パスは環境に合わせる）。`num_trials_per_task` は小さく。

```bash
python experiments/libero/evaluate.py \
  --ckpt_path pretrained/TurboVLA/checkpoints/libero/libero_object.pth \
  --stats_path experiments/libero/configs/libero_all4_stats.json \
  --stats_key libero_all4_no_noops \
  --task_suite_name libero_object \
  --num_trials_per_task 1 \
  --chunk_size 12 \
  --num_open_loop_steps 12 \
  --result_json_path outputs/evaluation/libero_object_smoke.json
```

| チェック | 結果 | メモ |
|----------|------|------|
| HF download 済み | | |
| evaluate が最後まで走った | | |
| result json のパス | | |


## 3. GPU 無しフォールバック（仍 Phase 4 完了可）

以下を自分の言葉で埋める。

1. `evaluate.py`（または Evo server）で必須だと思う引数 / 設定:
2. なぜ `chunk_size == num_open_loop_steps` か:
3. 使う予定の `stats_key`:
4. HF 上で確認した ckpt 名（メモ）:


In [ ]:
# ウォークスルー記録（自由記述）
walkthrough = {
    "required_args_or_settings": "",
    "why_chunk_equals_open_loop": "",
    "stats_key": "libero_all4_no_noops",
    "hf_ckpt_names_seen": [],
}
# 編集したら実行して確認
missing = [k for k, v in walkthrough.items() if v in ("", [], None)]
print("filled:" if not missing else f"still empty: {missing}")
for k, v in walkthrough.items():
    print(f"- {k}: {v}")


## 4. 完了判定

- [ ] Evo-1 server **または** TurboVLA eval **または** セクション 3 のウォークスルーを完了
- [ ] [quiz/q04](../../quiz/q04_upstream_smoke.md) を解いた
- [ ] [study/README.md](../README.md) の Phase 4 チェックを更新

次: [05_parc_transfer.md](../05_parc_transfer.md)
